# FLDT ApplicationDetails — Perimeter Table

Replicates the filtering effect of `INNER JOIN DWH.act.ApplicationDetails` from on-prem SP `[pld].[sp_Load_act_ApplicationDetails]`.

## How it works

* Single-column table: `{env}-cdh.derivativeinternal.act_applicationdetails` (ApplicationKey STRING)
* INSERT-only (SCD2 accumulation — keys never deleted)
* Daily growth: adds genuinely new ApplicationKeys (not in previous day's snapshot)
* Filter: `UserCreatedQualcoGroupCode <> 'C'`

## Execution order

1. Setup (env, run_date, prev_date)
2. Daily perimeter growth INSERT
3. Verify final state



In [0]:
import sys
sys.path.append("/Workspace/Shared")
import aws_utils

env = aws_utils.get_env_prefix()

run_date = str(spark.sql(f"""
    SELECT MAX(BusinessDate) FROM `{env}-cdh`.primaryactive.application WHERE SourceSystem = 'fldt'
""").collect()[0][0])

prev_date = str(spark.sql(f"""
    SELECT MAX(BusinessDate) FROM `{env}-cdh`.landed.fldt_application WHERE BusinessDate < {run_date!r}
""").collect()[0][0])

if run_date == "None":
    raise Exception("No current data in fldt_application")
if prev_date == "None":
    raise Exception("No previous day data in fldt_application")

print(f"env:        {env}")
print(f"run_date:   {run_date}")
print(f"prev_date:  {prev_date}")

In [0]:
result = spark.sql(f"""
    INSERT INTO `{env}-cdh`.derivativeinternal.act_applicationdetails
    SELECT DISTINCT fa.ApplicationKey
    FROM `{env}-cdh`.landed.fldt_application fa
    INNER JOIN (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY applicationid, customercode ORDER BY positiondate DESC) AS rn
        FROM `{env}-cdh`.landed.fldt_customer
        WHERE BusinessDate = {run_date!r}
    ) c ON c.applicationid = fa.id AND c.BusinessDate = fa.BusinessDate AND c.rn = 1
    WHERE fa.BusinessDate = {run_date!r}
      AND UPPER(COALESCE(fa.UserCreatedQualcoGroupCode, '')) <> 'C'
      AND NOT EXISTS (
          SELECT 1 FROM `{env}-cdh`.derivativeinternal.act_applicationdetails ad
          WHERE ad.ApplicationKey = fa.ApplicationKey
      )
      AND NOT EXISTS (
          SELECT 1 FROM `{env}-cdh`.landed.fldt_application fa_prev
          WHERE fa_prev.ApplicationKey = fa.ApplicationKey
            AND fa_prev.BusinessDate = {prev_date!r}
      )
""")
display(result)

In [0]:
df_verify = spark.sql(f"""
    SELECT 
        CURRENT_TIMESTAMP() AS run_timestamp,
        {run_date!r} AS business_date,
        (SELECT COUNT(*) FROM `{env}-cdh`.derivativeinternal.act_applicationdetails) AS total_perimeter_keys,
        (SELECT COUNT(DISTINCT app.ApplicationKey)
         FROM `{env}-cdh`.primaryactive.application app
         INNER JOIN `{env}-cdh`.derivativeinternal.act_applicationdetails ad
             ON ad.ApplicationKey = app.ApplicationKey
         WHERE app.SourceSystem = 'fldt'
           AND app.BusinessDate = {run_date!r}
        ) AS apps_matching_perimeter
""")
display(df_verify)